# 🌊 Basin Selection Study
## Which basin should I select for basin generalisation?

**R&D Notebook** — Analyses inter-basin similarity, LOBO performance across all 6 basins, pairwise transfer, and basin difficulty ranking.

---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — set QUICK_MODE=True to reduce compute (fewer epochs, 1D-only model)
# ══════════════════════════════════════════════════════════════════════════════
QUICK_MODE = True   # Set False for full-fidelity runs
LOBO_EPOCHS = 3 if QUICK_MODE else 15
PAIR_EPOCHS = 2 if QUICK_MODE else 8
USE_3D = not QUICK_MODE   # Skip heavy 3D convolutions in quick mode
USE_ENV = True
BATCH_SIZE = 64 if QUICK_MODE else 32

In [ ]:
import os, glob, math, time, warnings, copy
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from netCDF4 import Dataset as NC4Dataset
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, confusion_matrix, classification_report,
    f1_score, precision_recall_fscore_support
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.autograd import grad

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
matplotlib.rcParams['figure.dpi'] = 120

# ── Paths & Constants ─────────────────────────────────────────────────────────
PROJECT_ROOT = Path('/Users/thiruanand/MLfTCC')
DATA_ROOT    = PROJECT_ROOT / 'Notebooks' / 'data' / 'tropicyclonenet' / 'TCND_test'
BASINS = ['EP', 'NA', 'NI', 'SI', 'SP', 'WP']
BASIN_FULL = {'EP':'Eastern Pacific','NA':'North Atlantic','NI':'North Indian',
              'SI':'South Indian','SP':'South Pacific','WP':'Western Pacific'}
INT_NAMES = ['Weakening','Slow Weakening','Steady','Intensification']  # 4 classes from dataset (0-3)
DIR_NAMES = ['E','NE','N','NW','W','SW','S','SE']
RI_CLASS = 3  # Intensification is the highest class (class 3)

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED)

print(f"✓ Setup | Device: {DEVICE} | Quick: {QUICK_MODE} | Data: {DATA_ROOT.exists()}")

## 1. Data Loading
Reuse the data pipeline from the basin generalisation notebook.

In [ ]:
def scan_files(data_root, basins, subdir, ext):
    idx = {}
    for b in basins:
        d = data_root / subdir / b
        if not d.exists(): continue
        for f in sorted(d.rglob(f'*{ext}')):
            storm = f.parent.name if ext=='.npy' else f.stem.split('_')[1]
            dt = f.stem if ext=='.npy' else f.stem.split('_')[2]
            idx[(b, storm, dt)] = str(f)
    return idx

def build_env_vector(env_dict):
    def to_scalar(x):
        return np.float32(np.argmax(x)) if isinstance(x, np.ndarray) else np.float32(max(x, 0))
    return np.concatenate([
        np.array(env_dict.get('month', np.zeros(12)), dtype=np.float32),
        np.array(env_dict.get('area', np.zeros(6)), dtype=np.float32),
        np.array(env_dict.get('intensity_class', np.zeros(6)), dtype=np.float32),
        np.array([env_dict.get('wind', 0)], dtype=np.float32),
        np.array([env_dict.get('move_velocity', 0)], dtype=np.float32),
        np.array(env_dict.get('location_long', np.zeros(36)), dtype=np.float32),
        np.array(env_dict.get('location_lat', np.zeros(12)), dtype=np.float32),
        np.array([to_scalar(env_dict.get('history_direction12',-1))], dtype=np.float32),
        np.array([to_scalar(env_dict.get('history_direction24',-1))], dtype=np.float32),
        np.array([to_scalar(env_dict.get('history_inte_change24',-1))], dtype=np.float32),
    ])  # → (77,)

def load_data3d(nc_path):
    with NC4Dataset(nc_path, 'r') as nc:
        u = np.array(nc.variables['u'][:], dtype=np.float32)
        v = np.array(nc.variables['v'][:], dtype=np.float32)
        z_geo = np.array(nc.variables['z'][:], dtype=np.float32)
        sst_var = nc.variables['sst']
        sst_raw = np.ma.filled(sst_var[:], fill_value=np.nan).astype(np.float32)
    chs = [u[0,i] for i in range(4)] + [v[0,i] for i in range(4)] + [z_geo[0,i] for i in range(4)]
    chs.append(sst_raw[0] if sst_raw.ndim==3 else sst_raw)
    data_3d = np.stack(chs, axis=0)
    for c in range(13):
        ch = data_3d[c]
        valid = ch[~np.isnan(ch)]
        if len(valid) > 0:
            mu, std = valid.mean(), max(valid.std(), 1e-6)
            ch[np.isnan(ch)] = mu
            data_3d[c] = (ch - mu) / std
        else:
            data_3d[c] = np.zeros_like(ch)
    return data_3d

def compute_labeled_dataset(data_root, basins):
    """Load all samples with CORRECT column order and targets from Env-Data."""
    records = []
    d3d_idx = scan_files(data_root, basins, 'Data3D', '.nc')
    env_idx = scan_files(data_root, basins, 'Env-Data', '.npy')
    # Correct column order (from src/data/dataset.py):
    # ID | FLAG | LAT_norm | LONG_norm | WND_norm | PRES_norm | YYYYMMDDHH | Name
    TCND_COLS = ['ID', 'FLAG', 'LAT_norm', 'LONG_norm', 'WND_norm', 'PRES_norm', 'YYYYMMDDHH', 'Name']
    for basin in basins:
        d = data_root / 'Data1D' / basin / 'test'
        if not d.exists(): continue
        for f in sorted(d.glob('*.txt')):
            try:
                df = pd.read_csv(f, sep='\t', header=None, names=TCND_COLS)
                stem = f.stem
                year = stem[len(basin):len(basin)+4]
                tc_name = stem[len(basin)+4+3:]  # skip 'BST'
                for _, r in df.iterrows():
                    dt = str(int(r['YYYYMMDDHH']))
                    storm = tc_name
                    d3d_p = d3d_idx.get((basin,storm.upper(),dt)) or d3d_idx.get((basin,storm,dt))
                    env_p = env_idx.get((basin,storm.upper(),dt)) or env_idx.get((basin,storm,dt))
                    if not (d3d_p and env_p): continue
                    # Read TARGETS from Env-Data (NOT computed from wind change)
                    env_d = np.load(env_p, allow_pickle=True).item()
                    y_int = int(np.asarray(env_d.get('future_inte_change24', 2)).ravel()[0])
                    y_dir = int(np.asarray(env_d.get('future_direction24', 0)).ravel()[0])
                    if y_int < 0: y_int = 2  # sentinel -1 → steady
                    if y_dir < 0: y_dir = 0
                    y_int = min(y_int, 4)
                    y_dir = min(y_dir, 7)
                    # data1d: [LONG, LAT, PRES, WND] (matching src/data/dataset.py)
                    records.append({'basin':basin,'storm':storm,'datetime':dt,
                        'data1d':np.array([r['LONG_norm'],r['LAT_norm'],r['PRES_norm'],r['WND_norm']],dtype=np.float32),
                        'data3d_path':d3d_p,'env_path':env_p,'y_intensity':y_int,'y_direction':y_dir})
            except: pass
    return records

print("Building labeled dataset...")
t0 = time.time()
ALL_SAMPLES = compute_labeled_dataset(DATA_ROOT, BASINS)
print(f"✓ {len(ALL_SAMPLES):,} samples in {time.time()-t0:.1f}s")
basin_counts = Counter(s['basin'] for s in ALL_SAMPLES)
for b in BASINS:
    ri = sum(1 for s in ALL_SAMPLES if s['basin']==b and s['y_intensity']==4)
    print(f"  {b}: {basin_counts[b]:5d} | RI: {ri:3d} ({100*ri/max(basin_counts[b],1):.1f}%)")

## 2. Per-Basin Statistics

In [ ]:
# Build a comprehensive stats DataFrame
rows = []
for b in BASINS:
    samps = [s for s in ALL_SAMPLES if s['basin'] == b]
    n = len(samps)
    storms = set(s['storm'] for s in samps)
    int_counts = Counter(s['y_intensity'] for s in samps)
    dir_counts = Counter(s['y_direction'] for s in samps)
    ri_count = int_counts.get(RI_CLASS, 0)
    # Data1D stats
    d1d = np.array([s['data1d'] for s in samps])
    rows.append({
        'Basin': b, 'Full Name': BASIN_FULL[b],
        'Samples': n, 'Storms': len(storms),
        'RI Count': ri_count, 'RI Rate %': 100*ri_count/max(n,1),
        'Mean Lon': d1d[:,0].mean(), 'Mean Lat': d1d[:,1].mean(),
        'Mean Pres': d1d[:,2].mean(), 'Mean Wnd': d1d[:,3].mean(),
        'Std Wnd': d1d[:,3].std(),
        **{f'Int_{i}%': 100*int_counts.get(i,0)/max(n,1) for i in range(4)},
    })
stats_df = pd.DataFrame(rows)
display(stats_df.round(2))

In [ ]:
# Visualise class distribution per basin
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Intensity class distribution (stacked bar)
ax = axes[0]
bottom = np.zeros(len(BASINS))
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#c0392b']
for cls in range(4):
    vals = [sum(1 for s in ALL_SAMPLES if s['basin']==b and s['y_intensity']==cls) for b in BASINS]
    ax.bar(BASINS, vals, bottom=bottom, label=INT_NAMES[cls], color=colors[cls])
    bottom += vals
ax.set_title('Intensity Class Distribution per Basin', fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.set_ylabel('Count')

# RI rate comparison
ax = axes[1]
ri_rates = [stats_df[stats_df.Basin==b]['RI Rate %'].values[0] for b in BASINS]
bars = ax.bar(BASINS, ri_rates, color=sns.color_palette('husl', 6))
ax.set_title('RI Rate by Basin (%)', fontweight='bold')
ax.set_ylabel('RI Rate %')
for bar, val in zip(bars, ri_rates):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.1f}%', ha='center', fontsize=9)

plt.tight_layout(); plt.show()

## 3. Inter-Basin Distribution Similarity
Pairwise Jensen-Shannon divergence on intensity-class distributions.

In [ ]:
# Compute intensity-class probability distributions per basin
basin_int_dists = {}
for b in BASINS:
    samps = [s for s in ALL_SAMPLES if s['basin'] == b]
    counts = np.array([sum(1 for s in samps if s['y_intensity']==c) for c in range(5)], dtype=np.float64)
    basin_int_dists[b] = counts / counts.sum()

# Pairwise JSD
jsd_matrix = np.zeros((6, 6))
for i, b1 in enumerate(BASINS):
    for j, b2 in enumerate(BASINS):
        jsd_matrix[i, j] = jensenshannon(basin_int_dists[b1], basin_int_dists[b2])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(jsd_matrix, xticklabels=BASINS, yticklabels=BASINS,
            annot=True, fmt='.3f', cmap='YlOrRd', ax=ax)
ax.set_title('Jensen-Shannon Divergence (Intensity Distributions)', fontweight='bold')
plt.tight_layout(); plt.show()

print("\nMost similar pairs (lowest JSD):")
pairs = []
for i in range(6):
    for j in range(i+1, 6):
        pairs.append((BASINS[i], BASINS[j], jsd_matrix[i,j]))
for b1, b2, jsd in sorted(pairs, key=lambda x: x[2]):
    print(f"  {b1}↔{b2}: JSD={jsd:.4f}")

## 4. Feature-Space Similarity (t-SNE & PCA)
Embed Data1D + Env features and visualise basin clustering.

In [ ]:
# Collect feature vectors (Data1D + Env) for a subsample
MAX_PER_BASIN = 500
feature_vecs, basin_labels, int_labels = [], [], []

for b in BASINS:
    samps = [s for s in ALL_SAMPLES if s['basin']==b][:MAX_PER_BASIN]
    for s in samps:
        try:
            env = np.load(s['env_path'], allow_pickle=True).item()
            vec = np.concatenate([s['data1d'], build_env_vector(env)])
            feature_vecs.append(vec)
            basin_labels.append(b)
            int_labels.append(s['y_intensity'])
        except:
            pass

X = np.array(feature_vecs)
print(f"Feature matrix: {X.shape}")

# PCA
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X)

# t-SNE
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X)

# Silhouette score (basin clustering quality)
sil = silhouette_score(X, basin_labels)
print(f"Silhouette score (basin labels): {sil:.4f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
basin_colors = {b: c for b, c in zip(BASINS, sns.color_palette('husl', 6))}
c_list = [basin_colors[b] for b in basin_labels]

for ax, data, title in [(axes[0], X_pca, 'PCA'), (axes[1], X_tsne, 't-SNE')]:
    for b in BASINS:
        mask = [bl==b for bl in basin_labels]
        ax.scatter(data[mask, 0], data[mask, 1], label=b, s=8, alpha=0.5)
    ax.set_title(f'{title} — Colour by Basin (Silhouette={sil:.3f})', fontweight='bold')
    ax.legend(fontsize=8, markerscale=3)
    ax.set_xlabel(f'{title} 1'); ax.set_ylabel(f'{title} 2')

plt.tight_layout(); plt.show()

## 5. Model & Training Infrastructure
Shared model and training utilities (copied from base notebook).

In [ ]:
class ConvBnRelu(nn.Sequential):
    def __init__(self, in_c, out_c, k=3, s=1, p=1):
        super().__init__(nn.Conv2d(in_c,out_c,k,stride=s,padding=p,bias=False),
                         nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))

class SpatialEncoder(nn.Module):
    def __init__(self, in_ch=13, embed_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            ConvBnRelu(in_ch,32,k=7,s=2,p=3), ConvBnRelu(32,64,k=3,s=2,p=1),
            ConvBnRelu(64,128,k=3,s=2,p=1), ConvBnRelu(128,256,k=3,s=2,p=1),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.proj = nn.Sequential(nn.Linear(256, embed_dim), nn.LayerNorm(embed_dim))
    def forward(self, x): return self.proj(self.net(x))

class TrackEncoder(nn.Module):
    def __init__(self, in_dim=4, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim,64), nn.LayerNorm(64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64,embed_dim), nn.LayerNorm(embed_dim))
    def forward(self, x): return self.net(x)

class EnvEncoder(nn.Module):
    def __init__(self, in_dim=77, embed_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim,128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128,embed_dim), nn.LayerNorm(embed_dim))
    def forward(self, x): return self.net(x)

class PhysicsEncoder(nn.Module):
    def __init__(self, in_dim=8, phys_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim,64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(64,phys_dim), nn.LayerNorm(phys_dim))
    def forward(self, x): return self.net(x)

class MultimodalBackbone(nn.Module):
    def __init__(self, spatial_embed=128, track_embed=32, env_embed=64,
                 phys_dim=32, final_dim=128, use_3d=True, use_env=True):
        super().__init__()
        self.use_3d, self.use_env, self.phys_dim = use_3d, use_env, phys_dim
        self.spatial_enc = SpatialEncoder(embed_dim=spatial_embed) if use_3d else None
        self.track_enc = TrackEncoder(embed_dim=track_embed)
        self.env_enc = EnvEncoder(embed_dim=env_embed) if use_env else None
        self.phys_enc = PhysicsEncoder(phys_dim=phys_dim)
        fused_in = track_embed + (spatial_embed if use_3d else 0) + (env_embed if use_env else 0)
        self.projector = nn.Sequential(
            nn.Linear(fused_in, final_dim*2), nn.LayerNorm(final_dim*2), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(final_dim*2, final_dim), nn.LayerNorm(final_dim))
        self.env_dim = final_dim - phys_dim
        self.final_dim = final_dim
        self.phys_align = nn.Linear(phys_dim, phys_dim)

    def forward(self, batch):
        parts = []
        if self.use_3d: parts.append(self.spatial_enc(batch['data_3d']))
        parts.append(self.track_enc(batch['data_1d']))
        if self.use_env: parts.append(self.env_enc(batch['env_data']))
        z_full = self.projector(torch.cat(parts, dim=-1))
        z_env = z_full[:, :self.env_dim]
        z_phys_raw = z_full[:, self.env_dim:]
        phys_enc_out = self.phys_enc(batch['phys_features'])
        z_phys = z_phys_raw + self.phys_align(phys_enc_out)
        z = torch.cat([z_env, z_phys], dim=-1)
        return {'z':z, 'z_phys':z_phys, 'z_phys_raw':z_phys_raw, 'z_env':z_env}

class TaskHeads(nn.Module):
    def __init__(self, in_dim=128, n_int=4, n_dir=8):
        super().__init__()
        self.int_head = nn.Sequential(nn.Linear(in_dim,64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64,n_int))
        self.dir_head = nn.Sequential(nn.Linear(in_dim,64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64,n_dir))
    def forward(self, z): return self.int_head(z), self.dir_head(z)

class TCModel(nn.Module):
    def __init__(self, use_3d=True, use_env=True, phys_dim=32, final_dim=128):
        super().__init__()
        self.backbone = MultimodalBackbone(use_3d=use_3d, use_env=use_env,
                                           phys_dim=phys_dim, final_dim=final_dim)
        self.heads = TaskHeads(in_dim=final_dim)
    def forward(self, batch):
        feat = self.backbone(batch)
        li, ld = self.heads(feat['z'])
        return {**feat, 'logits_intensity':li, 'logits_direction':ld}

print("✓ Model architecture defined")

In [ ]:
class TCNDDataset(Dataset):
    def __init__(self, samples, use_3d=True, use_env=True):
        self.samples = samples
        self.use_3d = use_3d
        self.use_env = use_env
        self.basin_to_idx = {b:i for i,b in enumerate(BASINS)}
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        item = {
            'data_1d': torch.tensor(s['data1d'], dtype=torch.float32),
            'y_intensity': torch.tensor(s['y_intensity'], dtype=torch.long),
            'y_direction': torch.tensor(s['y_direction'], dtype=torch.long),
            'basin_idx': torch.tensor(self.basin_to_idx[s['basin']], dtype=torch.long),
        }
        if self.use_3d:
            item['data_3d'] = torch.tensor(load_data3d(s['data3d_path']), dtype=torch.float32)
        else:
            item['data_3d'] = torch.zeros(13, 81, 81)
        if self.use_env:
            env = np.load(s['env_path'], allow_pickle=True).item()
            item['env_data'] = torch.tensor(build_env_vector(env), dtype=torch.float32)
        else:
            item['env_data'] = torch.zeros(77)
        env_d = np.load(s['env_path'], allow_pickle=True).item() if self.use_env else {}
        sst_anom = s['data1d'][3] * 0.5
        wind_shear = abs(s['data1d'][2] - s['data1d'][3])
        lat_rad = abs(s['data1d'][1]) * 0.3
        coriolis = 2 * 7.2921e-5 * np.sin(np.clip(lat_rad, -np.pi/2, np.pi/2))
        mpi_proxy = max(0, sst_anom * 2 - wind_shear)
        bl_moisture = max(0, 1.0 - abs(s['data1d'][2]))
        outflow_temp = -0.5 * abs(s['data1d'][1])
        steering = s['data1d'][0] * 0.1
        current_int = s['data1d'][3]
        item['phys_features'] = torch.tensor(
            [sst_anom, wind_shear, coriolis, mpi_proxy,
             bl_moisture, outflow_temp, steering, current_int], dtype=torch.float32)
        return item

def make_lobo_split(all_samples, target_basin, val_frac=0.15):
    source = [s for s in all_samples if s['basin'] != target_basin]
    target = [s for s in all_samples if s['basin'] == target_basin]
    np.random.shuffle(source)
    n_val = int(len(source) * val_frac)
    return source[n_val:], source[:n_val], target

def make_per_basin_loaders(samples, batch_size=32, use_3d=True, use_env=True):
    by_basin = defaultdict(list)
    for s in samples: by_basin[s['basin']].append(s)
    loaders = {}
    for b, samps in by_basin.items():
        ds = TCNDDataset(samps, use_3d=use_3d, use_env=use_env)
        safe_bs = min(batch_size, len(ds))
        loaders[b] = DataLoader(ds, batch_size=safe_bs, shuffle=True, drop_last=True, num_workers=0)
    return loaders

def task_loss(li, ld, yi, yd, iw=1.0, dw=0.5):
    return iw * F.cross_entropy(li, yi) + dw * F.cross_entropy(ld, yd)

class ERM:
    def compute_loss(self, batches, model):
        losses = []
        for b in batches.values():
            out = model(b)
            losses.append(task_loss(out['logits_intensity'], out['logits_direction'],
                                   b['y_intensity'], b['y_direction']))
        erm = torch.stack(losses).mean()
        return erm, {'erm_loss': erm.item()}

def train_epoch(model, method, train_loaders, optimizer, device, extra_params=None):
    model.train()
    if isinstance(method, nn.Module): method.train()
    env_iters = {b: iter(l) for b, l in train_loaders.items()}
    n_batches = max(len(l) for l in train_loaders.values())
    epoch_loss = 0
    for _ in range(n_batches):
        batches = {}
        for b, it in env_iters.items():
            try: batch = next(it)
            except StopIteration:
                env_iters[b] = iter(train_loaders[b]); batch = next(env_iters[b])
            batches[b] = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k,v in batch.items()}
        optimizer.zero_grad()
        loss, metrics = method.compute_loss(batches, model)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if extra_params: nn.utils.clip_grad_norm_(extra_params, 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / n_batches

def evaluate(model, loader, device):
    model.eval()
    preds_int, labels_int = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k,v in batch.items()}
            out = model(batch)
            preds_int.extend(out['logits_intensity'].argmax(-1).cpu().numpy())
            labels_int.extend(batch['y_intensity'].cpu().numpy())
    pi, li = np.array(preds_int), np.array(labels_int)
    acc_int = float((pi == li).mean())
    weighted_f1 = float(f1_score(li, pi, average='weighted', zero_division=0))
    macro_f1    = float(f1_score(li, pi, average='macro', zero_division=0))
    prec, rec, f1, _ = precision_recall_fscore_support(li, pi, labels=[RI_CLASS], zero_division=0)
    ri_f1 = float(f1[0])
    return {'acc_int': acc_int, 'weighted_f1': weighted_f1, 'macro_f1': macro_f1, 'ri_f1': ri_f1}

def run_lobo(all_samples, target_basin, epochs, use_3d, use_env, bs, lr=5e-4):
    """Run single LOBO experiment, return results dict."""
    phys_dim = 32 if use_3d else 16
    final_dim = 128 if use_3d else (96 if use_env else 48)
    train_s, val_s, test_s = make_lobo_split(all_samples, target_basin)
    train_loaders = make_per_basin_loaders(train_s, bs, use_3d, use_env)
    val_loader = DataLoader(TCNDDataset(val_s, use_3d, use_env), batch_size=bs, num_workers=0)
    test_loader = DataLoader(TCNDDataset(test_s, use_3d, use_env), batch_size=bs, num_workers=0)
    model = TCModel(use_3d=use_3d, use_env=use_env, phys_dim=phys_dim, final_dim=final_dim).to(DEVICE)
    method = ERM()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best_val_wf1, best_state = -1, None
    for ep in range(1, epochs+1):
        t0 = time.time()
        loss = train_epoch(model, method, train_loaders, optimizer, DEVICE)
        val_r = evaluate(model, val_loader, DEVICE)
        if val_r['weighted_f1'] > best_val_wf1:
            best_val_wf1 = val_r['weighted_f1']
            best_state = copy.deepcopy(model.state_dict())
        print(f"  [LOBO→{target_basin}] Ep {ep:2d}/{epochs} | loss={loss:.4f} | "
              f"acc={val_r['acc_int']:.3f} | wF1={val_r['weighted_f1']:.3f} | "
              f"RI-F1={val_r['ri_f1']:.3f} | {time.time()-t0:.1f}s")
    if best_state: model.load_state_dict(best_state)
    # Source performance
    src_accs = []
    for b, loader in train_loaders.items():
        r = evaluate(model, loader, DEVICE)
        src_accs.append(r['acc_int'])
    src_acc = np.mean(src_accs)
    test_r = evaluate(model, test_loader, DEVICE)
    btg = (src_acc - test_r['acc_int']) / max(src_acc, 1e-8)
    return {'target': target_basin, 'src_acc': src_acc, **test_r, 'btg': btg, 'model': model}

print("✓ Training utilities defined")

## 6. LOBO Benchmark (All 6 Basins)
Run ERM LOBO with each basin as the held-out target.

In [ ]:
LOBO_RESULTS = []

for target in BASINS:
    print(f"\n{'='*60}")
    print(f"LOBO TARGET: {BASIN_FULL[target]} ({target})")
    print(f"{'='*60}")
    r = run_lobo(ALL_SAMPLES, target, epochs=LOBO_EPOCHS, use_3d=USE_3D, use_env=USE_ENV, bs=BATCH_SIZE)
    LOBO_RESULTS.append(r)
    print(f"  → Tgt Acc={r['acc_int']:.3f} | wF1={r['weighted_f1']:.3f} | RI-F1={r['ri_f1']:.3f} | BTG={r['btg']:.3f}")

# Summary table
print(f"\n{'='*90}")
print(f"{'Target':<8} {'Src Acc':>8} {'Tgt Acc':>8} {'wF1':>6} {'macF1':>6} {'RI-F1':>6} {'BTG':>8}")
print(f"{'-'*90}")
for r in LOBO_RESULTS:
    print(f"{r['target']:<8} {r['src_acc']:>8.3f} {r['acc_int']:>8.3f} "
          f"{r['weighted_f1']:>6.3f} {r['macro_f1']:>6.3f} {r['ri_f1']:>6.3f} {r['btg']:>8.3f}")
print(f"{'='*90}")

In [ ]:
# Visualise LOBO results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = sns.color_palette('husl', 6)

# Target accuracy
ax = axes[0]
accs = [r['acc_int'] for r in LOBO_RESULTS]
bars = ax.bar(BASINS, accs, color=colors)
ax.set_title('Target Basin Accuracy (LOBO)', fontweight='bold')
ax.set_ylabel('Accuracy')
for bar, val in zip(bars, accs): ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{val:.3f}', ha='center', fontsize=9)

# BTG
ax = axes[1]
btgs = [r['btg'] for r in LOBO_RESULTS]
bars = ax.bar(BASINS, btgs, color=colors)
ax.set_title('Basin Transfer Gap (lower = easier)', fontweight='bold')
ax.set_ylabel('BTG')
for bar, val in zip(bars, btgs): ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{val:.3f}', ha='center', fontsize=9)

# RI-F1
ax = axes[2]
ri_f1s = [r['ri_f1'] for r in LOBO_RESULTS]
bars = ax.bar(BASINS, ri_f1s, color=colors)
ax.set_title('RI F1-Score (LOBO)', fontweight='bold')
ax.set_ylabel('RI F1')
for bar, val in zip(bars, ri_f1s): ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{val:.3f}', ha='center', fontsize=9)

plt.tight_layout(); plt.show()

## 7. Pairwise Transfer Matrix
Train on basin A → evaluate on basin B for all 30 ordered pairs.

In [ ]:
# Pairwise transfer: Train on A, test on B
transfer_acc = np.zeros((6, 6))
transfer_ri_f1 = np.zeros((6, 6))

phys_dim = 16
final_dim = 48 if not USE_3D else 128

for i, src_basin in enumerate(BASINS):
    src_samps = [s for s in ALL_SAMPLES if s['basin'] == src_basin]
    # Train on source basin
    np.random.shuffle(src_samps)
    n_val = int(len(src_samps) * 0.15)
    train_s, val_s = src_samps[n_val:], src_samps[:n_val]
    
    train_ds = TCNDDataset(train_s, use_3d=USE_3D, use_env=USE_ENV)
    safe_bs = min(BATCH_SIZE, len(train_ds))
    train_loader = DataLoader(train_ds, batch_size=safe_bs, shuffle=True, drop_last=True, num_workers=0)
    train_loaders = {src_basin: train_loader}
    
    model = TCModel(use_3d=USE_3D, use_env=USE_ENV, phys_dim=phys_dim, final_dim=final_dim).to(DEVICE)
    method = ERM()
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    
    for ep in range(PAIR_EPOCHS):
        train_epoch(model, method, train_loaders, optimizer, DEVICE)
    
    # Evaluate on all basins
    for j, tgt_basin in enumerate(BASINS):
        tgt_samps = [s for s in ALL_SAMPLES if s['basin'] == tgt_basin]
        tgt_loader = DataLoader(TCNDDataset(tgt_samps, use_3d=USE_3D, use_env=USE_ENV),
                               batch_size=BATCH_SIZE, num_workers=0)
        r = evaluate(model, tgt_loader, DEVICE)
        transfer_acc[i, j] = r['acc_int']
        transfer_ri_f1[i, j] = r['ri_f1']
    
    print(f"Trained on {src_basin}: " + " | ".join(f"{b}={transfer_acc[i,j]:.3f}" for j, b in enumerate(BASINS)))

print("\n✓ Pairwise transfer complete")

In [ ]:
# Visualise transfer matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, mat, title in [(axes[0], transfer_acc, 'Accuracy'), (axes[1], transfer_ri_f1, 'RI F1-Score')]:
    sns.heatmap(mat, xticklabels=BASINS, yticklabels=BASINS,
                annot=True, fmt='.3f', cmap='RdYlGn', ax=ax, vmin=0, vmax=1)
    ax.set_title(f'Pairwise Transfer — {title}', fontweight='bold')
    ax.set_xlabel('Target Basin'); ax.set_ylabel('Source Basin')

plt.tight_layout(); plt.show()

## 8. Basin Difficulty Ranking & Recommendations

In [ ]:
# Compute difficulty metrics
print("═" * 80)
print("BASIN DIFFICULTY RANKING")
print("═" * 80)

# Average transfer gap when basin is the TARGET
avg_incoming_acc = []
for j, tgt in enumerate(BASINS):
    incoming = [transfer_acc[i, j] for i in range(6) if i != j]
    avg_incoming_acc.append((tgt, np.mean(incoming), np.std(incoming)))

print("\n1. Average accuracy when basin is TARGET (harder = lower):")
for b, m, s in sorted(avg_incoming_acc, key=lambda x: x[1]):
    print(f"   {b} ({BASIN_FULL[b]}): {m:.3f} ± {s:.3f}")

# Average transfer gap when basin is the SOURCE
avg_outgoing_acc = []
for i, src in enumerate(BASINS):
    outgoing = [transfer_acc[i, j] for j in range(6) if i != j]
    avg_outgoing_acc.append((src, np.mean(outgoing), np.std(outgoing)))

print("\n2. Average accuracy when basin is SOURCE (better source = higher):")
for b, m, s in sorted(avg_outgoing_acc, key=lambda x: -x[1]):
    print(f"   {b} ({BASIN_FULL[b]}): {m:.3f} ± {s:.3f}")

# Best source for each target
print("\n3. Best source basin for each target:")
for j, tgt in enumerate(BASINS):
    best_i = max([i for i in range(6) if i != j], key=lambda i: transfer_acc[i, j])
    print(f"   {tgt}: best source = {BASINS[best_i]} (acc={transfer_acc[best_i, j]:.3f})")

# LOBO-based ranking
print("\n4. LOBO ranking (by BTG — higher = harder to generalise to):")
for r in sorted(LOBO_RESULTS, key=lambda x: -x['btg']):
    print(f"   {r['target']} ({BASIN_FULL[r['target']]}): BTG={r['btg']:.3f}, Tgt Acc={r['acc_int']:.3f}")

print("\n" + "═" * 80)
print("RECOMMENDATIONS")
print("═" * 80)

# Sort by BTG descending (hardest first)
ranked = sorted(LOBO_RESULTS, key=lambda x: -x['btg'])
hardest = ranked[0]['target']
easiest = ranked[-1]['target']

print(f"""
• Hardest target basin: {hardest} ({BASIN_FULL[hardest]}) — highest BTG, most challenging for generalisation
  → Recommended as PRIMARY holdout basin for stress-testing domain generalisation methods.

• Easiest target basin: {easiest} ({BASIN_FULL[easiest]}) — lowest BTG
  → Useful as a sanity-check baseline.

• Recommended evaluation protocol:
  1. Use {hardest} as the main holdout target (hardest generalisation)
  2. Use {ranked[1]['target']} as a secondary target 
  3. Report results across ALL basins for comprehensive evaluation
""")

print("✅ Basin Selection Study complete!")